# Pasado `gpu-hybrid`: kompletna Colab verifikacija

Pre pokretanja izaberi **Runtime > Change runtime type > GPU**.

Notebook klonira tačan commit `c2a2c657`, proverava real-cubic selector matricu, zatim pokreće quick i full CPU/CUDA verifikaciju. Ako quick provera padne, izvršavanje se namerno zaustavlja pre full matrice.

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Uključi GPU: Runtime > Change runtime type > GPU"

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)

In [ ]:
import os
import shutil
import subprocess

REPO = "/content/Pasado"
URL = "https://github.com/dimitrijepesic/Pasado.git"
EXPECTED = "c2a2c657d9249b084d8fc0a91c67d683adfda86e"

shutil.rmtree(REPO, ignore_errors=True)
clone = subprocess.run([
    "git", "-c", "http.version=HTTP/1.1", "clone",
    "--branch", "gpu-hybrid", "--single-branch", URL, REPO,
], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(clone.stdout)

if clone.returncode != 0:
    print("Standardni clone nije uspeo; pokušavam fetch fallback.")
    shutil.rmtree(REPO, ignore_errors=True)
    os.makedirs(REPO)
    subprocess.run(["git", "-C", REPO, "init"], check=True)
    subprocess.run([
        "git", "-C", REPO, "-c", "http.version=HTTP/1.1",
        "fetch", "--depth", "1", URL, "refs/heads/gpu-hybrid",
    ], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "--detach", "FETCH_HEAD"], check=True)

head = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
print("HEAD:", head)
assert head == EXPECTED, f"Pogrešan commit: {head}"
assert not subprocess.check_output(["git", "-C", REPO, "status", "--short"], text=True).strip()
os.chdir(REPO)

In [ ]:
%pip install -q scikit-learn

In [ ]:
import numpy
import sklearn
import torch
import torchvision

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("numpy:", numpy.__version__)
print("sklearn:", sklearn.__version__)
print("GPU:", torch.cuda.get_device_name(0))

## 1. Device plumbing

Proverava svih 16 selector kombinacija sa realnim kubnim rešavačem. Ćelija mora završiti sa `PASS`.

In [ ]:
import sys

subprocess.run([
    sys.executable,
    "experiments/check_device_plumbing.py",
    "--device", "cuda",
    "--real-cubic",
    "--selector-matrix",
], cwd=REPO, check=True)

## 2. Quick matrica

Pet slika i epsilon indeksi 0, 8 i 15 za sve četiri mreže. Koristi dtype-specifične pragove (`1e-9` za float64 i `1e-5` za float32) i proverava CUDA determinizam, full/precise-only i očekivanu divergenciju starog kompleksnog rešavača.

In [ ]:
quick = subprocess.run([
    "bash", "-lc",
    """
    set -o pipefail
    PYTHONUNBUFFERED=1 python experiments/colab_verify_gpu_hybrid.py \
      --quick \
      --float64-limit 1e-9 \
      --float32-limit 1e-5 \
      --out /content/gpu_hybrid_quick.csv \
      2>&1 | tee /content/gpu_hybrid_quick.log
    """,
], cwd=REPO)

QUICK_OK = quick.returncode == 0
print("Quick exit:", quick.returncode)
assert QUICK_OK, "Quick provera je pala. Ne pokreći full; preuzmi quick log i CSV."

In [ ]:
import pandas as pd

quick_df = pd.read_csv("/content/gpu_hybrid_quick.csv")
display(quick_df.sort_values("max_relative_difference", ascending=False))
assert (quick_df["result"] == "PASS").all()
print("QUICK: ALL PASS")

## 3. Full matrica

Pokreće se samo posle potpuno zelenog quick rezultata. Obuhvata 30 slika i svih 16 epsilona za 3layer, 4layer, 5layer i big.

In [ ]:
assert QUICK_OK, "Quick nije prošao."

full = subprocess.run([
    "bash", "-lc",
    """
    set -o pipefail
    PYTHONUNBUFFERED=1 python experiments/colab_verify_gpu_hybrid.py \
      --float64-limit 1e-9 \
      --float32-limit 1e-5 \
      --out /content/gpu_hybrid_full.csv \
      2>&1 | tee /content/gpu_hybrid_full.log
    """,
], cwd=REPO)

FULL_OK = full.returncode == 0
print("Full exit:", full.returncode)
assert FULL_OK, "Full CUDA matrica ima FAIL rezultat."

In [ ]:
full_df = pd.read_csv("/content/gpu_hybrid_full.csv")
display(full_df.sort_values("max_relative_difference", ascending=False))
assert (full_df["result"] == "PASS").all()
print("FULL: ALL PASS")
print(full_df.groupby(["network", "result"]).size())

## 4. Preuzimanje rezultata

Ova ćelija može da se pokrene ručno i ako quick ili full provera padne; arhivira sve rezultate koji postoje.

In [ ]:
from zipfile import ZipFile
from google.colab import files

paths = [
    "/content/gpu_hybrid_quick.csv",
    "/content/gpu_hybrid_quick.log",
    "/content/gpu_hybrid_full.csv",
    "/content/gpu_hybrid_full.log",
]
archive = "/content/gpu_hybrid_results.zip"

with ZipFile(archive, "w") as output:
    for path in paths:
        if os.path.exists(path):
            output.write(path, arcname=os.path.basename(path))

print("Arhivirani fajlovi:", [path for path in paths if os.path.exists(path)])
files.download(archive)